In [1]:
! pip install ipywidgets
! pip install torch torchvision diffusers transformers accelerate safetensors

In [2]:
import torch
from diffusers import StableDiffusionControlNetImg2ImgPipeline, ControlNetModel
from PIL import Image

In [3]:
device = (
    torch.device("cuda") if torch.cuda.is_available()
    else torch.device("mps") if torch.backends.mps.is_available()
    else torch.device("cpu")
)
print(f"Using device: {device}")

Using device: mps


In [4]:
# Load your controlnet model (choose the type)
# Popular options: canny, depth, scribble, openpose, lineart
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/control_v11p_sd15_scribble",   # good for sketches
    torch_dtype=torch.float16
).to(device)

In [5]:
# Load the base SD model
pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=torch.float16
).to(device)

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

`torch_dtype` is deprecated! Use `dtype` instead!


In [6]:
# Load your input image
init_image = Image.open("front-garden/front_garden.jpeg").convert("RGB")

width, height = init_image.size
init_image = init_image.resize((width//3, height//3))

In [7]:
# Load the sketch/explaining image
sketch = Image.open("front-garden/planting_plan.jpeg").convert("RGB")

sketch = sketch.resize((width//3, height//3))

In [8]:
# Set your prompt
prompt = "Include the plants as outlined in the sketch"

In [9]:
# Run img2img + sketch conditioning
image = pipe(
    prompt=prompt,
    image=init_image,     # img2img source
    control_image=sketch, # sketch/guide image
    strength=0.6,
    guidance_scale=7.5,
    num_inference_steps=50
).images[0]

  0%|          | 0/30 [00:00<?, ?it/s]

In [10]:
# Save the generated image
image.save("result_controlnet_img2img.png")

print("Image saved as result_controlnet_img2img.png")

Image saved as result_controlnet_img2img.png
